主要实现的功能：
 - 生成销占比程序表
 - 生成销售预估参数

In [1]:
import pandas as pd
import os

In [2]:
# 定义通用的函数

def concate(*args):
    ret = args[0]
    for i in args[1:]:
        ret += ';' + i
    return ret

def replace(x):
    x = str(x)
    x = x.replace('[', '')
    x = x.replace(']', '')
    x = x.replace("'", '')
    return x


def build_and_merge(df, column_name, new_column_name):
    df['temp'] = df['站点'] + '：' + df[column_name]

    locations = df[['站点', '积加SKU', 'temp']]
    locations.drop_duplicates(inplace=True)
    locations = locations.pivot_table(index=['积加SKU'], values='temp', aggfunc=concate)['temp']
    locations = locations.reset_index(level=[0])
    locations['temp'] = locations['temp'].apply(replace)
    locations.rename(columns={'temp': new_column_name}, inplace=True)

    df.drop(columns=['temp'], inplace=True)
    print(locations.head())
    # print(f'columns：{locations.columns}')

    return pd.merge(left=df, right=locations, how='left', left_on=['积加SKU'], right_on=['积加SKU'])

In [3]:
# 初始化通用变量
path = '../src_data/'

# 1. 销售预估表合并

In [4]:
sales_predict_files = [f for f in os.listdir(f'{path}销售预估参数') if f.endswith('.xlsx')]

sales_predict = pd.concat(
    [pd.read_excel(f'{path}销售预估参数/{file}', sheet_name='销量预估表-parameter') for file in sales_predict_files],
    ignore_index=True
)

sales_predict.to_excel(f'{path}Listing预估表-模板.xlsx', sheet_name='parms', index=False)


In [5]:
sales_predict.columns

Index(['站点', 'Listing', '月份', 'Listing_月度预估销量'], dtype='object')

# 2. 销占比参数表处理

## 2.1销占比参数合并

In [6]:
sales_files = [f for f in os.listdir(f'{path}销占比参数') if f.endswith('.xlsx')]

# usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
#        'SKU销占比', '需求定位', '发货定位', '备货定位',
#        '店铺', '下单天数确认','总发货天数']

# usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
#        'SKU销占比', '需求定位', '发货定位', '备货定位',
#        '店铺', '运营保底下单天数','总发货天数','快递', '空运', '海运']

usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
       'SKU销占比', '需求定位', '发货定位', '备货定位',
       '店铺', '运营保底下单天数','总发货天数','快递','空运','海运', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间']

sales = pd.concat(
    [pd.read_excel(f'{path}销占比参数/{file}', sheet_name='parameter') for file in sales_files],
    ignore_index=True
)

sales = sales[usecols]

In [7]:
sales.columns

Index(['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
       'SKU销占比', '需求定位', '发货定位', '备货定位', '店铺', '运营保底下单天数', '总发货天数', '快递', '空运',
       '海运', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间'],
      dtype='object')

In [8]:
# sales.to_excel('C:/Users/gongy/Downloads/销占比/Listing销占比程序表.xlsx')

## 2.2 导入产品信息
[点击查看最新产品信息](https://www.kdocs.cn/l/civjbw2s04Be)

In [11]:
usecols_for_info = ['MSKU', '店铺/站点', '品牌', 'FNSKU', '仓储类型', '一级品类', '品类A', '品类B']
info = pd.read_excel(f'{path}/产品信息_20231103.xlsx', usecols=usecols_for_info)

In [12]:
# 唯一性检测，如果存在重复数据则 主动抛出异常
duplicated_cnt = info.duplicated(subset=['MSKU', '店铺/站点']).sum()

if duplicated_cnt > 0:
    duplicated_values = info[info.duplicated(subset=['MSKU', '店铺/站点'], keep=False)][['MSKU', '店铺/站点']]
    raise ValueError(f"检测到产品信息表中存在重复MSKU：\n{duplicated_values}")

info.drop_duplicates(subset=['MSKU', '店铺/站点'], inplace=True)
info.rename(columns={'店铺/站点': '店铺-站点'}, inplace=True)

## 2.2 构建可能用到的维度

In [13]:
# 拼接 店铺+站点 用于鱼皮FBA库存 SENWAYZON:CA
sales['店铺-站点'] = sales['店铺'] + ':' + sales['站点']

# 获取 'FNSKU', '仓储类型', '一级品类', '品类A', '品类B'
sales = pd.merge(left=sales, right=info, how='left', on=['店铺-站点', 'MSKU'])

# sales.to_excel('../file/parms_process_sales-原.xlsx', index = False)

In [14]:
# 使用通用函数构建 定位-集合
sales = build_and_merge(sales, '规模定位','定位-集合')
# sales.to_excel('../file/parms_process_sales-定位-集合.xlsx', index = False)
# 使用通用函数构建 需求定位-集合
sales = build_and_merge(sales, '需求定位','需求定位-集合')
# sales.to_excel('../file/parms_process_sales-需求定位-集合.xlsx', index = False)

                        积加SKU              定位-集合
0   832-1 Black Ice Blue Lens  DE：中尾 UK：短尾 US：短尾
1  832-1 Black Ice Green Lens  DE：中尾 UK：中尾 US：中尾
2    832-1 Black Ice Red Lens  DE：中尾 UK：中尾 US：中尾
3             832-1 BlackGrey  DE：短尾 UK：短尾 US：短尾
4             832-1 BlackPink  DE：长尾 UK：中尾 US：长尾
                        积加SKU            需求定位-集合
0   832-1 Black Ice Blue Lens  DE：维护 UK：维护 US：维护
1  832-1 Black Ice Green Lens  DE：维护 UK：维护 US：维护
2    832-1 Black Ice Red Lens  DE：维护 UK：维护 US：维护
3             832-1 BlackGrey  DE：维护 UK：维护 US：维护
4             832-1 BlackPink  DE：废弃 UK：废弃 US：维护


C:\Users\gongy\AppData\Local\Temp\ipykernel_29524\578780657.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  locations.drop_duplicates(inplace=True)
C:\Users\gongy\AppData\Local\Temp\ipykernel_29524\578780657.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  locations.drop_duplicates(inplace=True)


In [ ]:
# 输出预览
# sales.query('积加SKU=="RG605 Black L"')
sales.head()

In [15]:
sales.to_excel('../src_data/Listing销占比程序表-模板.xlsx', index=False, sheet_name='parms')

In [ ]:
sales.columns

In [ ]:
# # 初始化通用变量
# path = 'C:\\Users\\gongy\\Downloads\\'

In [ ]:
# sales_files = [f for f in os.listdir(f'{path}销占比') if f.endswith('.xlsx')]

# # usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
# #        'SKU销占比', '需求定位', '发货定位', '备货定位',
# #        '店铺', '下单天数确认','总发货天数']

# usecols = ['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
#        'SKU销占比', '需求定位', '发货定位', '备货定位',
#        '店铺', '运营保底下单天数','总发货天数']

# sales = pd.concat(
#     [pd.read_excel(f'{path}销占比/{file}', sheet_name='parameter') for file in sales_files],
#     ignore_index=True
# )


In [ ]:
# sales.to_excel('C:\\Users\\gongy\\Downloads\\销占比/Listing销占比程序表.xlsx', index=False, sheet_name='parms')